# 🧹 Day 3: 脏数据清洗 & RFM 模型实战

> **场景**: 昨天我们 dealt with 完美的模拟数据。但现实世界的数据是**脏乱差**的。
> 今天我们要扮演“数据保洁阿姨”，先把垃圾清理干净，然后进行一次高大上的 **RFM 用户分层分析**。
>
> **核心技能**:
> 1. **缺失值**: `fillna`, `dropna` (相当于 SQL `COALESCE`, `IS NOT NULL`)
> 2. **重复值**: `drop_duplicates` (相当于 SQL `DISTINCT`)
> 3. **类型转换**: `astype`, `pd.to_datetime` (相当于 SQL `CAST`)
> 4. **文本清洗**: `.str.replace`, `.str.split`
> 5. **🔥 RFM 分层**: `qcut` (自动分箱)

## ⛽️ 模块 0: 今日函数加油站 (Cheat Sheet)

| 函数 | 作用 | 常用参数 | SQL 类比 |
| :--- | :--- | :--- | :--- |
| **`dropna()`** | 删除缺失值 | `subset=['col']` (只查某列) | `WHERE col IS NOT NULL` |
| **`fillna(v)`** | 填充缺失值 | `value=0` (填0) | `COALESCE(col, 0)` |
| **`drop_duplicates()`** | 删除重复行 | `subset=['id']`, `keep='last'` | `DISTINCT` / `ROW_NUMBER` |
| **`astype()`** | 类型转换 | `int`, `float`, `str` | `CAST(col AS type)` |
| **`pd.to_datetime()`** | 转日期 | `errors='coerce'` (错的变空) | `TRY_CAST` |
| **`.str`** | 字符串操作 | `.str.replace()`, `.str.split()` | `REPLACE`, `SPLIT` |
| **`pd.qcut()`** | 自动分箱 | `q=3` (3等分), `labels=['A','B']` | `NTILE(3)` |

In [1]:
import pandas as pd
import numpy as np

# 🗑️ 准备一份“脏”数据 (Dirty Data)
raw_data = {
    'order_id': ['ORD-101', 'ORD-102', 'ORD-103', 'ORD-102', 'ORD-104', 'ORD-105', np.nan, 'ORD-106'],
    'user_id': [1, 2, 1, 2, 3, 4, 5, 6],
    'amount': ['$100.5', '$20.0', '50.0', '$20.0', '300', None, '50.0', '120.0'], # 混杂着 $ 符号和 None
    'date': ['2023-01-01', '2023-01-02', '2023/01/01', '2023-01-02', '2023-01-05', '2023-01-06', '2023-01-07', 'invalid_date'],
    'status': ['Paid', 'Paid', 'Pending', 'Paid', 'Paid', 'Refunded', 'Paid', 'Paid']
}

df = pd.DataFrame(raw_data)
print("🤢 感受一下这份脏数据：")
df

🤢 感受一下这份脏数据：


,order_id,user_id,amount,date,status
0,ORD-101,1,$100.5,2023-01-01,Paid
1,ORD-102,2,$20.0,2023-01-02,Paid
2,ORD-103,1,50.0,2023/01/01,Pending
3,ORD-102,2,$20.0,2023-01-02,Paid
4,ORD-104,3,300,2023-01-05,Paid
5,ORD-105,4,None,2023-01-06,Refunded
6,NaN,5,50.0,2023-01-07,Paid
7,ORD-106,6,120.0,invalid_date,Paid


---

## 🧱 Part 1: 清洗流水线 (The Cleaning Pipeline)

在做分析之前，必须先把地扫干净。请按照步骤完成清洗。

### 🧹 任务 1: 处理缺失值与重复项
1. **去重**: `order_id` 有重复的 (ORD-102)，请把完全重复的行删掉。
2. **去空**: `order_id` 为空的行是无效数据，请直接删掉。
3. **填空**: `amount` 里的缺失值 (None)，请用 **0** 填充。

In [2]:
# [你的代码]
# 提示：drop_duplicates(),dropna(subset=...), fillna()
cleaned_df = df.drop_duplicates(subset=['order_id']).dropna(subset=['order_id'])
# df.dropna(subset=['order_id'])
cleaned_df['amount']=cleaned_df['amount'].fillna('0')
cleaned_df

,order_id,user_id,amount,date,status
0,ORD-101,1,$100.5,2023-01-01,Paid
1,ORD-102,2,$20.0,2023-01-02,Paid
2,ORD-103,1,50.0,2023/01/01,Pending
4,ORD-104,3,300,2023-01-05,Paid
5,ORD-105,4,0,2023-01-06,Refunded
7,ORD-106,6,120.0,invalid_date,Paid


### 🧹 任务 2: 类型大清洗 (Type Casting)
现在的 `amount` 是 `object` (因为有 '$' 符号)，`date` 也是 `object`(字符串)。

1. **清洗金额**: 去掉 '$' 符号，转为 `float` 类型。
2. **清洗日期**: 将 `date` 列转为标准的 `datetime` 类型（非法日期会自动变成 NaT，不用管它）。

In [3]:
# [你的代码]
# 💡 提示：如果发现日期变成了 NaT，说明之前的运行可能弄脏了数据。
# 请尝试点击菜单栏的 ⏩ (Run All) 或者 "Restart Kernel" 重跑一遍整个 Notebook。

# 1. 💰 清洗金额
# 技巧：先 .astype(str) 强转字符串，防止因为已经是数字而报错
cleaned_df['amount'] = cleaned_df['amount'].astype(str).str.replace('$', '').astype(float)

# 2. 📅 清洗日期
# ✅ 终极修正：format='mixed'
# 原因：数据里混用了 2023-01-01 (横杠) 和 2023/01/01 (斜杠)。
# Pandas 2.0+ 默认会猜一种格式套用到底，猜错了就报错。加上 'mixed' 告诉它可以“这就这那样”。
cleaned_df['date'] = pd.to_datetime(cleaned_df['date'], format='mixed', errors='coerce')

cleaned_df


,order_id,user_id,amount,date,status
0,ORD-101,1,100.5,2023-01-01,Paid
1,ORD-102,2,20.0,2023-01-02,Paid
2,ORD-103,1,50.0,2023-01-01,Pending
4,ORD-104,3,300.0,2023-01-05,Paid
5,ORD-105,4,0.0,2023-01-06,Refunded
7,ORD-106,6,120.0,NaT,Paid


---

## 🔥 Part 2: 综合实战 - RFM 用户分层

现在数据干净了，我们要开始做真正的分析了。
**目标**: 根据用户的购买行为，给用户打上 **"高价值" / "一般" / "流失"** 的标签。

我们简化一下 RFM 模型，只看 **M (Monetary - 消费金额)** 和 **F (Frequency - 消费频次)**。

### 📊 步骤 1: 计算指标 (Aggregation)
请按 `user_id` 分组，算出每个用户的：
1. **M值**: 总消费金额 (`sum`)
2. **F值**: 订单数量 (`count`)

存成一个新的 DataFrame，叫 `rfm_df`。

In [4]:
# [你的代码]
# rfm_df = {'user_id','M_score','F_score'}
# rfm_df = pd.DataFrame(rfm_df)
# rfm_df = cleaned_df['user_id'].unique().reset_index()
rfm_df = cleaned_df['amount'].groupby(cleaned_df['user_id']).agg(['sum','count']).reset_index()
# rfm_df['F_score'] = cleaned_df['amount'].groupby(cleaned_df['user_id']).count()
rfm_df.columns = ['user_id', 'M_score', 'F_score']
rfm_df

,user_id,M_score,F_score
0,1,150.5,2
1,2,20.0,1
2,3,300.0,1
3,4,0.0,1
4,6,120.0,1


### 🧐 步骤 2: 探索数据分布 (Exploration)
**这是你之前提到的重点**：不要直接拍脑袋定阈值，先看看大家到底花了多少钱。

请使用 `describe()` 查看 `rfm_df` 的统计分布。

In [5]:
# [你的代码]
rfm_df.describe()


,user_id,M_score,F_score
count,5.000000,5.000000,5.000000
mean,3.200000,118.100000,1.200000
std,1.923538,120.116818,0.447214
min,1.000000,0.000000,1.000000
25%,2.000000,20.000000,1.000000
50%,3.000000,120.000000,1.000000
75%,4.000000,150.500000,1.000000
max,6.000000,300.000000,2.000000


### 🏷️ 步骤 3: 自动分箱打标 (Scoring)
假设我们不想手动定死 "大于 100 是 VIP"，而是希望 **"前 33% 的人是 VIP"** (动态分层)。

Pandas 有个神器叫 `pd.qcut` (Quantile Cut)，它可以自动按排名切分。

**任务**：
1. 将用户的 **M值** 分成 3 等份，标签分别设为 `['Low', 'Mid', 'High']`。
2. 把这个新标签加到 `rfm_df` 里，列名叫 `level`。

In [6]:
# [你的代码]
# 提示：pd.qcut(rfm_df['amount'], q=3, labels=['Low', 'Mid', 'High'])
rfm_df['level']=pd.qcut(rfm_df['M_score'], q=3, labels=['Low', 'Mid', 'High'])
rfm_df

,user_id,M_score,F_score,level
0,1,150.5,2,High
1,2,20.0,1,Low
2,3,300.0,1,High
3,4,0.0,1,Low
4,6,120.0,1,Mid


### 🔍 步骤 4: 谁是高价值用户？
筛选出所有 `level == 'High'` 的用户 ID。

In [7]:
# [你的代码]
rfm_df[rfm_df['level'] == 'High']['user_id']    


0    1
2    3
Name: user_id, dtype: int64

---
### 💡 参考答案 (点击展开)

In [8]:
# Part 1 清洗
df = df.drop_duplicates()
df = df.dropna(subset=['order_id'])
df['amount'] = df['amount'].fillna(0)

df['amount'] = df['amount'].astype(str).str.replace('$', '').astype(float)
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# Part 2 RFM
rfm_df = df.groupby('user_id')['amount'].agg(['sum', 'count'])
rfm_df.columns = ['M_score', 'F_score']
# describe() 看分布
print(rfm_df.describe())
# qcut 自动分箱
rfm_df['level'] = pd.qcut(rfm_df['M_score'], q=3, labels=['Low', 'Mid', 'High'])
print(rfm_df[rfm_df['level'] == 'High'])

          M_score   F_score
count    5.000000  5.000000
mean   118.100000  1.200000
std    120.116818  0.447214
min      0.000000  1.000000
25%     20.000000  1.000000
50%    120.000000  1.000000
75%    150.500000  1.000000
max    300.000000  2.000000
         M_score  F_score level
user_id                        
1          150.5        2  High
3          300.0        1  High
